## Silver Layer - Data Cleaning & Transformation

This notebook processes raw data from the Bronze layer and applies data cleaning and transformation logic.

Key operations performed:
- Removal of duplicate records
- Handling of missing/null values
- Data type standardization (timestamps, numeric fields)
- Date and timestamp normalization
- Feature engineering (delivery time, late delivery flag, etc.)
- Preparation of data for downstream analytics

The output of this layer is a set of clean, structured Delta tables ready for the Gold layer.

### Set Catalog and Schema

In [0]:
from pyspark.sql.functions import *

spark.sql("USE CATALOG e_commerce")
spark.sql("USE SCHEMA silver")


Clean Customers

In [0]:
# Customers Table


customers = spark.table("e_commerce.bronze.bronze_customers") \
    .dropDuplicates(["customer_id"]) \
    .fillna({
        "customer_city": "unknown",
        "customer_state": "unknown"
    }) \
    .withColumn("customer_city", lower(trim(col("customer_city")))) \
    .withColumn("customer_state", lower(trim(col("customer_state")))) \
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int"))

customers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_customers")


display(customers)

Clean Geolocation

In [0]:
# Geolocation Table 
geolocation = spark.table("e_commerce.bronze.bronze_geolocation") \
    .dropDuplicates(["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"]) \
    .fillna({
        "geolocation_city": "unknown",
        "geolocation_state": "unknown"
    }) \
    .withColumn("geolocation_city", lower(trim(col("geolocation_city")))) \
    .withColumn("geolocation_state", lower(trim(col("geolocation_state")))) \
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("int")) \
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double")) \
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))

geolocation.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_geolocation")


display(geolocation)


Clean Order_Items


In [0]:
#Order_items Table

order_items = spark.table("e_commerce.bronze.bronze_order_items") \
    .dropDuplicates(["order_id", "order_item_id"]) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

order_items.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_order_items")
display(order_items)

Clean Payments

In [0]:
#Payments Table
payments = spark.table("e_commerce.bronze.bronze_order_payments") \
    .dropDuplicates(["order_id", "payment_sequential"]) \
    .withColumn("payment_type", lower(trim(col("payment_type")))) \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double"))

payments.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_order_payments")

display(payments)

Clean Reviews

In [0]:
# Reviews Table
reviews = spark.table("e_commerce.bronze.bronze_order_reviews") \
    .dropDuplicates(["review_id"]) \
    .filter(col("review_score").isNotNull()) \
    .withColumn(
        "review_creation_date",
        coalesce(
            to_timestamp(col("review_creation_date"), "M/d/yyyy H:mm"),
            to_timestamp(col("review_creation_date"), "yyyy-MM-dd HH:mm:ss")
        )
    ) \
    .withColumn(
        "review_answer_timestamp",
        coalesce(
            to_timestamp(col("review_answer_timestamp"), "M/d/yyyy H:mm"),
            to_timestamp(col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss")
        )
    ) \
    .withColumn("review_score", col("review_score").cast("int")) \
    .withColumn(
        "is_low_rating",
        when(col("review_score") < 3, 1).otherwise(0)
    )

reviews.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_reviews")

display(reviews)

Clean Orders

In [0]:
#Orders Table

orders = spark.table("e_commerce.bronze.bronze_orders") \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .filter(col("order_purchase_timestamp").isNotNull()) \
    .withColumn("order_approved_at",
                coalesce(col("order_approved_at"), col("order_purchase_timestamp"))) \
    .filter(
        col("order_delivered_customer_date").isNull() |
        (col("order_delivered_customer_date") >= col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "delivery_time_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "is_late_delivery",
        when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0)
    ) \
    .withColumn(
        "is_delivered",
        when(col("order_status") == "delivered", 1).otherwise(0)
    )

orders.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_status") \
    .saveAsTable("e_commerce.silver.silver_orders")
display(orders)


Clean Products

In [0]:
#Products Table 
products = spark.table("e_commerce.bronze.bronze_products") \
    .dropDuplicates(["product_id"]) \
    .fillna({
        "product_category_name": "unknown"
    }) \
    .withColumn("product_weight_g", col("product_weight_g").cast("int")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("int")) \
    .withColumn(
        "is_category_missing",
        when(col("product_category_name") == "unknown", 1).otherwise(0)
    )

products.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_products")

display(products)

Clean Sellers


In [0]:
# Sellers_Table
sellers = spark.table("e_commerce.bronze.bronze_sellers") \
    .dropDuplicates(["seller_id"]) \
    .fillna({
        "seller_city": "unknown",
        "seller_state": "unknown"
    }) \
    .withColumn("seller_city", lower(trim(col("seller_city")))) \
    .withColumn("seller_state", lower(trim(col("seller_state")))) \
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int"))

sellers.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("e_commerce.silver.silver_sellers")
display(sellers)